### グラフデータ読み込み

In [42]:
# Community-verification/util/io.py をimportするためのコード
# io.pyを編集した場合に、Jupyter Notebook上で再読み込みするために必要
import importlib
import os
import sys
sys.path.append(os.path.abspath("..")) 
from util import io
importlib.reload(io)

<module 'util.io' from '/Users/kaoriogawa/研究/community-detection-social-networks-dev/Community-verification/util/io.py'>

In [43]:
import networkx as nx
from pathlib import Path

name = "LFR01"

path = Path("../../data") / name

# データ読み込み
G = nx.read_edgelist(path / "edgelist.txt.gz", create_using=nx.DiGraph(), nodetype=int)
communities = io.communitydict_to_communitylist(io.read_communitydict(path / "communities.txt.gz"))

In [58]:
print(num_nodes := G.number_of_nodes())
print(num_edges := G.number_of_edges())

5000
28582


### シミュレーションを試運転するためのBAモデル作成＆実験設定

In [9]:
import json
import random
import networkx as nx
import os

def create_ba_directed_graph(n:int, m:int, seed:int, output_file:str, bidirectional:bool):
    """BAモデルをベースにした有向グラフを生成し、エッジリスト（u v）形式で出力する関数

    引数:
        n (int): ノード数
        m (int): 新規ノード追加時に貼るエッジ数
        seed (int): 乱数シード
        output_file (str): 保存するテキストファイルのパス
        bidirectional (bool): Trueの場合は全エッジを双方向に、Falseの場合は確率50%で一方向化
    """

    output_dir = os.path.dirname(output_file)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    
    # 1. BAモデルで無向グラフを作成
    g_undirected = nx.barabasi_albert_graph(n=n, m=m, seed=seed)

    # 2. 有向グラフに変換
    if bidirectional:
        g_directed = g_undirected.to_directed()
    else:
        g_directed = nx.DiGraph()
        random.seed(seed)
        for u, v in g_undirected.edges():
            if random.random() < 0.5:
                g_directed.add_edge(u, v)
            else:
                g_directed.add_edge(v, u)

    # 3. エッジリストを書き出し
    with open(output_file, "w", encoding="utf-8") as f:
        for u, v in g_directed.edges():
            f.write(f"{u} {v}\n")

    print(
        f"グラフ生成完了: ノード数={g_directed.number_of_nodes()}, エッジ数={g_directed.number_of_edges()} -> {output_file}"
    )

In [10]:
def create_config_json(
    output_path:str, graph_path:str, false_users:list[int], cost_func:str, sample_size:int, iterations:list[dict]
):
    """評価プログラム用の設定JSONファイルを生成する関数

    引数:
        output_path (str): 保存先JSONファイルのパス
        graph_path (str): 読み込むグラフ構造ファイルのパス
        false_users (list[int]): 偽情報発信源のノードIDリスト
        cost_func (str): コスト関数 ('follower' または 'influence')
        sample_size (int): 伝播シミュレーションの試行回数
        iterations (list[dict]): イテレーション条件の辞書リスト
    """
    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    config_data = {
        "graph_path": graph_path,
        "false_users": false_users,
        "cost_func": cost_func,
        "sample_size": sample_size,
        "iterations": iterations,
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(config_data, f, indent=2)

    print(f"設定ファイル生成完了: -> {output_path}")

In [36]:
graph_file_path = "../../data/BA01/graph_ba.txt"
config_file_path = "../../data/BA01/config.json"

# 1. 有向グラフデータの生成（引数を決定して呼び出し）
create_ba_directed_graph(
    n=200,  # ノード数
    m=10, # エッジ生成数
    seed=42,  # シード値
    output_file="../../data/BA01/graph_ba.txt",  # 保存ファイル名
    bidirectional=False,  # ランダム一方向化
)

# 2. JSON設定ファイルの生成（引数を決定して呼び出し）
create_config_json(
    output_path="../../data/BA01/config.json",  # 出力JSON名
    graph_path="../data/BA01/graph_ba.txt",  # 使用するグラフファイル
    false_users=[0, 1],  # 偽情報発信源
    cost_func="user",  # コスト関数名
    sample_size=100,  # サンプル数
    iterations=[  # イテレーション条件
        {"seed": 0, "capacity": 1},
        {"seed": 0, "capacity": 2},
        {"seed": 0, "capacity": 3},
        {"seed": 0, "capacity": 4},
        {"seed": 1, "capacity": 1},
        {"seed": 1, "capacity": 2},
        {"seed": 1, "capacity": 3},
        {"seed": 1, "capacity": 4},
    ],
)

グラフ生成完了: ノード数=200, エッジ数=1900 -> ../../data/BA01/graph_ba.txt
設定ファイル生成完了: -> ../../data/BA01/config.json


In [ ]:
import os

# ノートブックがある場所から見たパスで作成
os.makedirs("../../data/BA01/output", exist_ok=True)

# フォルダが存在するか確認
print("outputフォルダ存在確認:", os.path.exists("../../data/BA01/output"))

outputフォルダ存在確認: True


In [37]:
import networkx as nx

# 1. エッジリストファイルを有向グラフ(DiGraph)として読み込み
# ※スペース/タブ区切りに対応。カンマ区切りの場合は delimiter=',' を追加してください。
G = nx.read_edgelist('../../data/BA01/graph_ba.txt', create_using=nx.DiGraph())

# 2. (u -> v) は存在するが (v -> u) は存在しない非対称なエッジを検索
unidirectional_edges = [(u, v) for u, v in G.edges() if not G.has_edge(v, u)]

# 3. 判定結果の出力
if unidirectional_edges:
    print(f"判定結果: 【有向グラフ】です。")
    print(f"一方向のみのエッジ数: {len(unidirectional_edges)} / 全エッジ数: {G.number_of_edges()}")
    print("一方向エッジの例 (先頭5件):", unidirectional_edges[:5])
else:
    print("判定結果: 【無向グラフ】と同等です（すべてのエッジが双方向に対称です）。")

判定結果: 【有向グラフ】です。
一方向のみのエッジ数: 1900 / 全エッジ数: 1900
一方向エッジの例 (先頭5件): [('1', '0'), ('1', '11'), ('1', '18'), ('1', '19'), ('1', '29')]


### 虚偽情報・真情報シード選定および実験設定（JSON）の自動生成スクリプト

In [44]:
import json
import random
import typing
from pathlib import Path
from typing import Dict, List, Literal, Optional, Tuple, Union

import networkx as nx

# ※ 独自のコミュニティ読み込みモジュール（io）がある場合はインポートしてください
# import io

# 共通ユーティリティ
def get_subgraph_and_node_set(
    G: nx.DiGraph, community_nodes: List[int]
) -> Tuple[nx.DiGraph, set]:
    """コミュニティノード集合と、それらに基づく誘発部分グラフを取得する helper 関数。"""
    nodes_set = set(community_nodes)
    sub_G = G.subgraph(nodes_set)
    return sub_G, nodes_set


# 虚偽情報の発信源選定関数
from typing import Dict, List, Literal, Optional, Tuple, Union, TypeAlias

BridgeMode: TypeAlias = Literal["v_only", "u_only", "both"]
NodeMethod: TypeAlias = Literal[
    "internal_degree", "internal_pagerank", "global_pagerank"
]

def select_bridge_seeds(
    G: nx.DiGraph,
    true_community_nodes: List[int],
    false_community_nodes: List[int],
    mode: BridgeMode = "v_only",
) -> List[int]:
    """境界（ブリッジ）エッジのノードを選定。"""
    true_set = set(true_community_nodes)
    false_set = set(false_community_nodes)

    u_count: Dict[int, int] = {}
    v_count: Dict[int, int] = {}

    for u_node, v_node in G.edges():
        if u_node in true_set and v_node in false_set:
            u_count[u_node] = u_count.get(u_node, 0) + 1
            v_count[v_node] = v_count.get(v_node, 0) + 1
        elif u_node in false_set and v_node in true_set:
            v_count[u_node] = v_count.get(u_node, 0) + 1
            u_count[v_node] = u_count.get(v_node, 0) + 1

    if not u_count or not v_count:
        raise ValueError("コミュニティ間に境界エッジが見つかりませんでした。")

    top_v = max(v_count.keys(), key=lambda n: v_count[n])
    top_u = max(u_count.keys(), key=lambda n: u_count[n])

    if mode == "v_only":
        return [top_v]
    elif mode == "u_only":
        return [top_u]
    elif mode == "both":
        return [top_v, top_u]
    else:
        raise ValueError(f"Unknown mode: {mode}")


def select_center_seeds(
    G: nx.DiGraph,
    false_community_nodes: List[int],
    method: NodeMethod = "internal_degree",
    top_k: int = 1,
) -> List[int]:
    """コミュニティの中心ノードを選定。"""
    sub_G, _ = get_subgraph_and_node_set(G, false_community_nodes)

    if method == "internal_degree":
        scores = {
            node: sub_G.in_degree(node) + sub_G.out_degree(node)
            for node in sub_G.nodes()
        }
    elif method == "internal_pagerank":
        scores = nx.pagerank(sub_G)
    elif method == "global_pagerank":
        global_pr = nx.pagerank(G)
        scores = {node: global_pr[node] for node in false_community_nodes}
    else:
        raise ValueError(f"Unknown method: {method}")

    sorted_nodes = sorted(scores.keys(), key=lambda n: scores[n], reverse=True)
    return sorted_nodes[:top_k]


def select_periphery_seeds(
    G: nx.DiGraph,
    false_community_nodes: List[int],
    num_seeds: int = 1,
    seed: Optional[int] = 42,
) -> List[int]:
    """コミュニティの周辺（端寄り）ノードを選定。"""
    sub_G, _ = get_subgraph_and_node_set(G, false_community_nodes)
    internal_degrees = {
        node: sub_G.in_degree(node) + sub_G.out_degree(node)
        for node in sub_G.nodes()
    }
    valid_candidates = [
        node for node, deg in internal_degrees.items() if deg >= 1
    ]

    if not valid_candidates:
        valid_candidates = list(false_community_nodes)

    min_deg = min(internal_degrees[node] for node in valid_candidates)
    periphery_candidates = [
        node for node in valid_candidates if internal_degrees[node] == min_deg
    ]

    if seed is not None:
        random.seed(seed)

    if len(periphery_candidates) >= num_seeds:
        selected_seeds = random.sample(periphery_candidates, num_seeds)
    else:
        selected_seeds = periphery_candidates

    return selected_seeds

In [48]:
def create_config_json(
    output_path: Union[str, Path],
    graph_path: str,
    false_users: List[int],
    cost_func: Literal["user", "influence"] = "user",
    sample_size: int = 1000,
    seeds: Optional[List[int]] = None,
    user_capacities: Optional[List[int]] = None,
    influence_capacities: Optional[List[int]] = None,
    iterations: Optional[List[Dict[str, int]]] = None,
) -> None:
    
    if iterations is None:
        if seeds is None:
            seeds = list(range(9))  # seed 0 ~ 8

        if user_capacities is None:
            user_capacities = [1, 2, 3, 4]

        if influence_capacities is None:
            influence_capacities = [20, 40, 60, 80]

        if cost_func == "user":
            capacities = user_capacities
        elif cost_func == "influence":
            capacities = influence_capacities
        else:
            raise ValueError(f"Unknown cost_func: {cost_func}")

        # seed × capacity の全組み合わせで iterations リストを作成
        iterations = [
            {"seed": seed, "capacity": cap}
            for seed in seeds
            for cap in capacities
        ]

    config_data = {
        "graph_path": graph_path,
        "false_users": false_users,
        "cost_func": cost_func,
        "sample_size": sample_size,
        "iterations": iterations,
    }

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(config_data, f, indent=2, ensure_ascii=False)

    print(f"設定ファイル保存完了: {output_path}")


In [54]:
def generate_all_scenarios(
    graph_name: str,
    true_comm_id: int,
    false_comm_id: int,
    periphery_multi_seeds: Optional[int] = None,
    periphery_seed: int = 42,
    cost_funcs: List[Literal["user", "influence"]] = ["user", "influence"],
    sample_size: int = 1000,
    seeds: Optional[List[int]] = None,
    user_capacities: Optional[List[int]] = None,
    influence_capacities: Optional[List[int]] = None,
    iterations: Optional[List[Dict[str, int]]] = None,
    data_dir: Union[str, Path] = "../../data",
    output_base_dir: Union[str, Path] = "../../data/exp_settings",
    priphery_multi_seeds: Optional[int] = None,  # 引数のタイポ表記互換用
) -> None:
    # すべての攻撃シナリオおよびコスト関数（user, influence）の組み合わせでJSONファイルを一括生成する関数。

    # タイポ表記 (priphery_multi_seeds) にも対応
    if periphery_multi_seeds is None:
        if priphery_multi_seeds is not None:
            periphery_multi_seeds = priphery_multi_seeds
        else:
            periphery_multi_seeds = 5

    if seeds is None:
        seeds = list(range(9))  # デフォルト: seed 0 ~ 8

    data_path = Path(data_dir) / graph_name

    G = nx.read_edgelist(
        data_path / "edgelist.txt.gz", create_using=nx.DiGraph(), nodetype=int
    )
    communities = io.communitydict_to_communitylist(
        io.read_communitydict(data_path / "communities.txt.gz")
    )

    true_nodes = communities[true_comm_id]
    false_nodes = communities[false_comm_id]

    # シナリオごとの false_users 決定
    scenarios: Dict[str, List[int]] = {}

    # ① ブリッジノードシナリオ (v_only, u_only, both)
    for kind in typing.get_args(BridgeMode):
        scenarios[f"bridge_{kind}"] = select_bridge_seeds(
            G, true_nodes, false_nodes, mode=kind
        )

    # ② 中心ノードシナリオ (internal_degree, internal_pagerank, global_pagerank)
    for method in typing.get_args(NodeMethod):
        scenarios[f"center_{method}"] = select_center_seeds(
            G, false_nodes, method=method
        )

    # ③ 端寄りノードシナリオ (single, multiple)
    scenarios["periphery_single"] = select_periphery_seeds(
        G, false_nodes, num_seeds=1, seed=periphery_seed
    )
    scenarios["periphery_multiple"] = select_periphery_seeds(
        G, false_nodes, num_seeds=periphery_multi_seeds, seed=periphery_seed
    )

    # シナリオ × コスト関数（"user", "influence"）ごとに JSON を保存
    for scenario_name, false_seeds in scenarios.items():
        for cost_func in cost_funcs:
            # コスト関数ごとのサブフォルダ（user / influence）を作成して出力
            output_dir = (
                Path(output_base_dir) / graph_name / f"f_comm_{false_comm_id}" / cost_func
            )
            output_path = output_dir / f"{scenario_name}.json"

            create_config_json(
                output_path=output_path,
                graph_path=f"../data/{graph_name}/edgelist.txt.gz",
                false_users=false_seeds,
                cost_func=cost_func,
                sample_size=sample_size,
                seeds=seeds,
                user_capacities=user_capacities,
                influence_capacities=influence_capacities,
            )

In [57]:
generate_all_scenarios(
    graph_name="LFR01",
    true_comm_id=2,
    false_comm_id=8,# 1,3,8
    priphery_multi_seeds=5,
    periphery_seed=42,
)

設定ファイル保存完了: ../../data/exp_settings/LFR01/f_comm_8/user/bridge_v_only.json
設定ファイル保存完了: ../../data/exp_settings/LFR01/f_comm_8/influence/bridge_v_only.json
設定ファイル保存完了: ../../data/exp_settings/LFR01/f_comm_8/user/bridge_u_only.json
設定ファイル保存完了: ../../data/exp_settings/LFR01/f_comm_8/influence/bridge_u_only.json
設定ファイル保存完了: ../../data/exp_settings/LFR01/f_comm_8/user/bridge_both.json
設定ファイル保存完了: ../../data/exp_settings/LFR01/f_comm_8/influence/bridge_both.json
設定ファイル保存完了: ../../data/exp_settings/LFR01/f_comm_8/user/center_internal_degree.json
設定ファイル保存完了: ../../data/exp_settings/LFR01/f_comm_8/influence/center_internal_degree.json
設定ファイル保存完了: ../../data/exp_settings/LFR01/f_comm_8/user/center_internal_pagerank.json
設定ファイル保存完了: ../../data/exp_settings/LFR01/f_comm_8/influence/center_internal_pagerank.json
設定ファイル保存完了: ../../data/exp_settings/LFR01/f_comm_8/user/center_global_pagerank.json
設定ファイル保存完了: ../../data/exp_settings/LFR01/f_comm_8/influence/center_global_pagerank.json
設定ファイル保存完了: ../.

In [ ]:
# 真情報シード選定

def select_true_seed_users(
    G: nx.DiGraph,
    community_nodes: List[int],
    top_k: int = 1,
    method: Literal["internal_degree", "pagerank"] = "internal_degree",
) -> List[int]:
    """真情報の発信源ノードを選定する。"""
    sub_G, _ = get_subgraph_and_node_set(G, community_nodes)

    if method == "internal_degree":
        scores = {
            node: sub_G.in_degree(node) + sub_G.out_degree(node)
            for node in sub_G.nodes()
        }
    elif method == "pagerank":
        scores = nx.pagerank(sub_G)
    else:
        raise ValueError(f"Unknown method: {method}")

    sorted_nodes = sorted(scores.keys(), key=lambda n: scores[n], reverse=True)
    return sorted_nodes[:top_k]
